# ZINDI Arabic Multi-Label Classification

## Imports



In [1]:
import hashlib
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## Env Variables

In [2]:
ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061a\u064b-\u0652]")
DATA_DIR = Path("/content")
ID_COLUMN = "ID"
NON_TEXT = re.compile(r"[^\w\s\u0600-\u06ff]")
OTHER_TARGET_COLUMNS = [
    "Geography",
    "Politics & Conflict",
    "Health & Wellbeing",
    "Science",
    "Sports",
]
OUTPUT_DIR = Path("/content/submissions")
QATAR_COLUMN = "Qatar Related"
SEED = 42
TARGET_COLUMNS = [QATAR_COLUMN] + OTHER_TARGET_COLUMNS
TEXT_COLUMN = "text"
WHITESPACE = re.compile(r"\s+")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Utils

In [3]:
def load_data(data_dir=DATA_DIR):
    """
    Loads train and test CSV files and validates required columns.
    """
    train_frame = pd.read_csv(data_dir / "Train.csv")
    test_frame = pd.read_csv(data_dir / "Test.csv")

    required_train = {ID_COLUMN, TEXT_COLUMN, *TARGET_COLUMNS}
    required_test = {ID_COLUMN, TEXT_COLUMN}

    missing_train = required_train.difference(train_frame.columns)
    missing_test = required_test.difference(test_frame.columns)

    if missing_train or missing_test:
        raise ValueError(f"Missing train columns: {missing_train}; missing test columns: {missing_test}")

    return train_frame, test_frame

In [4]:
def preprocess_arabic(text):
    """
    Normalizes Arabic text by removing diacritics and unifying character variants.
    """
    text = str(text)
    text = ARABIC_DIACRITICS.sub("", text)
    # Character normalization: Alif variants and Ta Marbuta
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي").replace("ة", "ه")
    return WHITESPACE.sub(" ", NON_TEXT.sub(" ", text)).strip()

In [5]:
def score_predictions(actual, predicted, target_columns=TARGET_COLUMNS):
    """
    Calculates weighted F1-score for each target and returns the mean.
    """
    scores = {
        column: f1_score(actual[column], predicted[column], average="weighted")
        for column in target_columns
    }
    scores["mean"] = float(np.mean(list(scores.values())))
    return scores

In [6]:
def make_submission(test_frame, predictions, output_path):
    """
    Formats predictions into the competition submission format and saves to CSV.
    """
    submission = test_frame[[ID_COLUMN]].copy()
    submission[TARGET_COLUMNS] = predictions[TARGET_COLUMNS].to_numpy()
    submission.to_csv(output_path, index=False)
    return submission

## Preprocess text and create a validation split

This section applies a specific Arabic normalizer that handles diacritics, punctuation, and letter variations (such as normalizing 'Alif' forms, 'Ya', and 'Ta Marbuta'). After cleaning, the data is split into training and validation sets, using a stratified approach on the `Qatar Related` target to ensure balanced representation across both subsets.

In [ ]:
# Load training and test dataframes
train_df, test_df = load_data()
print(f"Train shape: {train_df.shape}; test shape: {test_df.shape}")

# Display the first 2 rows of the training dataframe
display(train_df.head(2))

# Display count of missing values for text and target columns
display(train_df[[TEXT_COLUMN, *TARGET_COLUMNS]].isna().sum().to_frame("missing"))

# Display value counts for each target column to understand distribution
for column in TARGET_COLUMNS:
    print(f"\n{column}")
    display(train_df[column].value_counts(dropna=False).to_frame("count"))

Train shape: (3786, 8); test shape: (1262, 2)


,ID,text,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,fL8IqbB5haFSy2gk,كشف مسؤول رياضي مصري أن الميدالية الذهبية التي...,1,G-7,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-3
1,Jivkt5fbyLOBsaQd,أشاد وزير الدولة للشؤون الخارجية لدولة قطر، سل...,1,G-8,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS


,missing
text,0
Qatar Related,0
Geography,0
Politics & Conflict,0
Health & Wellbeing,0
Science,0
Sports,0



Qatar Related


,count
Qatar Related,
0,2005
1,1781


...

,count
Health & Wellbeing,
NO_HEALTH,3108
H-1,349
H-2,329



Science


,count
Science,
NO_SCIENCE,2605
SC-1,588
SC-3,321
SC-2,272



Sports


,count
Sports,
NO_SPORTS,2314
SP-3,461
SP-4,370
SP-2,338
SP-1,303


In [12]:
# Clean and normalize Arabic text for both training and test sets using custom rules
train_df["processed_text"] = train_df[TEXT_COLUMN].fillna("").map(preprocess_arabic)
test_df["processed_text"] = test_df[TEXT_COLUMN].fillna("").map(preprocess_arabic)

# Split the dataset into 80% training and 20% validation
# We use 'stratify' on the binary 'Qatar Related' labels to maintain consistent class proportions
train_part, validation_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df[QATAR_COLUMN],
)
print(f"Training rows: {len(train_part)}; validation rows: {len(validation_part)}")

Training rows: 3028; validation rows: 758


## TF-IDF & LightGBM Baseline

To establish a baseline, we use a **One-vs-Rest** strategy for multi-label classification. The process involves two main steps:

1.  **Feature Extraction (TF-IDF):** We convert the processed Arabic text into numerical vectors using Term Frequency-Inverse Document Frequency. We use both unigrams and bigrams (`ngram_range=(1, 2)`) to capture context, and limit the features to the top 100,000 most frequent terms.
2.  **Classification (LightGBM):** We train a separate LightGBM classifier for each target column. LightGBM is chosen for its efficiency with sparse TF-IDF features.

Performance is measured using the **Weighted F1-score** for each category, and the final metric is the mean of these scores, as required by the competition.

In [9]:
# Initialize TF-IDF Vectorizer for n-grams
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000)
train_features = vectorizer.fit_transform(train_part["processed_text"])
validation_features = vectorizer.transform(validation_part["processed_text"])
test_features = vectorizer.transform(test_df["processed_text"])

In [10]:
validation_predictions = pd.DataFrame(index=validation_part.index)
test_predictions = pd.DataFrame(index=test_df.index)
tfidf_models = {}

# One-vs-Rest strategy: Train one LightGBM model per target label
for column in TARGET_COLUMNS:
    model = LGBMClassifier(
        n_estimators=50,
        learning_rate=0.08,
        num_leaves=31,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1
    )
    model.fit(train_features, train_part[column])
    tfidf_models[column] = model

    validation_predictions[column] = model.predict(validation_features)
    test_predictions[column] = model.predict(test_features)

# Evaluate baseline performance
results = score_predictions(validation_part[TARGET_COLUMNS], validation_predictions[TARGET_COLUMNS])
print(results)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

{'Qatar Related': 0.8813591963529179, 'Geography': 0.7061466727631173, 'Politics & Conflict': 0.8442355486747818, 'Health & Wellbeing': 0.9153081275416356, 'Science': 0.8917850360505338, 'Sports': 0.9019643986561272, 'mean': 0.856799830006519}


In [11]:
tfidf_submission = make_submission(
    test_df,
    test_predictions,
    OUTPUT_DIR / "submission_tfidf.csv",
)
display(tfidf_submission.head())

,ID,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,jiRadWgPKa2i51Xl,0,G-3,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-2
1,IoKhm6KjwCkwEnAZ,0,G-4,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
2,ZihSiR1hnDU6w7Lz,1,G-3,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
3,X72YJR7IVU7CCKbh,1,G-4,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-4
4,TpxFAGsHEekF2MpS,0,G-11,PC-1,H-1,NO_SCIENCE,NO_SPORTS
